## Preprocessing

In [1]:
import mne
import numpy as np
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import glob
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
# Set dataset path
dataset_path = Path('/home/aloo/CNS_Summer_Project_1/Infants_data')

# Get list of all subjects
subjects = sorted([d for d in os.listdir(dataset_path) if d.startswith('sub-NORB')])
print(f"Found {len(subjects)} subjects")

Found 103 subjects


In [3]:
# Function to find EDF files for a subject
def find_edf_files(subject_id):
    """Find all EDF files for a given subject."""
    subject_path = dataset_path / subject_id
    return sorted(subject_path.rglob('*.edf')) if subject_path.exists() else []

# Test with first subject
test_subject = subjects[0]
edf_files = find_edf_files(test_subject)
print(f"\n{test_subject}:")
print(f"  Found {len(edf_files)} EDF file(s)")
for edf in edf_files:
    print(f"  - {edf.name}")


sub-NORB00001:
  Found 1 EDF file(s)
  - sub-NORB00001_ses-1_task-EEG_eeg.edf


In [4]:
# Function to load EDF file
def load_edf_data(edf_path):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)  
    print(f"\nLoaded: {Path(edf_path).name}")    
    return raw

# Load the first EDF file as a test
if edf_files:
    raw = load_edf_data(edf_files[0])
else:
    print("No EDF files found for testing")


Loaded: sub-NORB00001_ses-1_task-EEG_eeg.edf


In [5]:
# Detect and remove bad channels
def detect_bad_channels(raw, flat_threshold=1e-12, noise_threshold=1e-2):
    data = raw.get_data()
    channel_names = raw.ch_names
    bad_refs = {'PG1', 'PG2', 'REF', 'GND', 'GROUND', '25+', '26+', '27+'}
    stds = np.std(data, axis=1)
    ranges = np.ptp(data, axis=1)
    return [
        ch for ch, s, r in zip(channel_names, stds, ranges)
        if s < flat_threshold or s > noise_threshold or r == 0 or ch.upper() in bad_refs
    ]

# Detect bad channels for test subject-session
if 'raw' in locals():
    bad_channels = detect_bad_channels(raw)
    raw_clean = raw.copy()
    
    if bad_channels:
        raw_clean.drop_channels(bad_channels)
        print(f"\nRemoved {len(bad_channels)} bad channel(s)")
        print(f"Remaining channels ({len(raw_clean.ch_names)}): {raw_clean.ch_names}")
    else:
        print("\nNo bad channels detected. Proceeding with all channels.")

else:
    print("Error: 'raw' data not found. Please run Step 1 first.")


Removed 2 bad channel(s)
Remaining channels (19): ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'CZ', 'PZ']


In [6]:
# Segment data based on annotation file
def load_annotations(subject_id, session_id, dataset_path):
    """
    Load annotation file for a given subject and session.
    """
    annotations_path = (
        dataset_path / 'derivatives' / 'NeuronicEEG' / subject_id /
        session_id / 'eeg' / f'{subject_id}_{session_id}_task-EEG_annotations.tsv'
    )
    if not annotations_path.exists():
        print(f"Warning: No annotations file found at {annotations_path}")
        return None

    rows = []
    for line in annotations_path.read_text().splitlines()[1:]:
        fields = [f.strip() for f in line.split('\t')]
        if len(fields) >= 2 and fields[0] and fields[1]:
            try:
                rows.append([
                    float(fields[0]),
                    float(fields[1]),
                    fields[2] if len(fields) > 2 and fields[2] else 'eyes_closed'
                ])
            except ValueError:
                pass

    if not rows:
        print(f"Warning: No valid data in annotations file: {annotations_path}")
        return None

    return pd.DataFrame(rows, columns=['onset', 'duration', 'label'])

def segment_data_from_annotations(raw, annotations_df, label='eyes_closed'):
    sfreq = raw.info['sfreq']
    data = raw.get_data()
    n_samples = data.shape[1]

    label_lower = label.lower().strip()
    ann_labels = annotations_df['label'].fillna('').astype(str).str.lower().str.strip()

    if label_lower == 'eyes_closed':
        mask = ann_labels.isin(['eyes_closed', 'ojos_cerrados'])
    else:
        mask = ann_labels == label_lower

    segments = []
    segment_info = []

    for _, row in annotations_df[mask].iterrows():
        start = int(row['onset'] * sfreq)
        end = int((row['onset'] + row['duration']) * sfreq)
        if 0 <= start <= end <= n_samples:
            segments.append(data[:, start:end])
            segment_info.append({
                'onset': row['onset'],
                'duration': row['duration'],
                'start_sample': start,
                'end_sample': end
            })
        else:
            print(f"Warning: Segment at onset={row['onset']}s exceeds data bounds, skipping")

    print(f"\nExtracted {len(segments)} valid segments")
    return segments, segment_info

# Apply segmentation based on test subject-session
if 'raw_clean' in locals():
    if 'edf_files' in locals() and edf_files:
        parts = edf_files[0].stem.split('_')
        subject_id = parts[0]
        session_id = next((p for p in parts if p.startswith('ses')), 'ses-1')

        annotations_df = load_annotations(subject_id, session_id, dataset_path)
        if annotations_df is None:
            print("Could not load annotations. Skipping segmentation.")
        else:
            segments, segment_info = segment_data_from_annotations(
                raw_clean, annotations_df, label='eyes_closed'
            )
            print(
                f"\nSuccessfully segmented data for {subject_id} {session_id}"
                if segments else
                f"\nNo valid segments found for {subject_id} {session_id}"
            )
    else:
        print("Error: EDF file information not found.")
else:
    print("Error: 'raw_clean' data not found. Please run bad channel detection first.")


Extracted 32 valid segments

Successfully segmented data for sub-NORB00001 ses-1


## Pearson correlation calculation for each segment of each session of each subject
* Not doing modulus of the values here, saving negative values as they are

In [7]:
# Batch compute Pearson correlation matrices for every segment of every subject-session
output_base = Path('/home/aloo/CNS_Summer_Project_1/correlation_matrices_segment_wise_(notAbsolute)')
output_base.mkdir(parents=True, exist_ok=True)


def save_segment_correlation_matrix(matrix, channel_labels, subject_id, session_id, segment_idx, output_base):
    session_output = output_base / subject_id / session_id
    session_output.mkdir(parents=True, exist_ok=True)
    out_path = session_output / f'{subject_id}_{session_id}_segment-{segment_idx:02d}_pearson_corr.csv'
    df = pd.DataFrame(matrix, index=channel_labels, columns=channel_labels)
    df.to_csv(out_path)
    return out_path


def process_all_subject_sessions(dataset_path, output_base, label='eyes_closed'):
    subjects = sorted([d for d in os.listdir(dataset_path) if d.startswith('sub-NORB')])

    for subject_id in subjects:
        subject_path = dataset_path / subject_id
        edf_files = sorted(subject_path.rglob('*.edf'))
        if not edf_files:
            print(f'Skipping {subject_id}: no EDF files found')
            continue

        for edf_path in edf_files:
            edf_name = edf_path.stem
            parts = edf_name.split('_')
            session_id = next((p for p in parts if p.startswith('ses')), 'ses-1')

            print(f'\nProcessing {subject_id} {session_id} from {edf_path.name}')
            raw = load_edf_data(edf_path)

            bad_channels = detect_bad_channels(raw)
            raw_clean = raw.copy()
            if bad_channels:
                raw_clean.drop_channels(bad_channels)
                print(f'  Removed bad channels: {bad_channels}')

            annotations_df = load_annotations(subject_id, session_id, dataset_path)
            if annotations_df is None:
                print(f'  No annotation file for {subject_id} {session_id}, skipping')
                continue

            segments, segment_info = segment_data_from_annotations(raw_clean, annotations_df, label=label)
            if not segments:
                print(f'  No valid segments for {subject_id} {session_id}')
                continue

            channel_labels = raw_clean.ch_names
            for idx, segment in enumerate(segments, start=1):
                corr_matrix = np.corrcoef(segment)
                if corr_matrix.shape != (19, 19):
                    print(f'  Warning: segment {idx} has {corr_matrix.shape[0]} channels after cleaning')

                out_path = save_segment_correlation_matrix(corr_matrix, channel_labels, subject_id, session_id, idx, output_base)
                print(f'  Saved segment {idx} matrix: {out_path.name} (shape={corr_matrix.shape})')

    print(f'\nBatch processing complete. Saved results under: {output_base}')


process_all_subject_sessions(dataset_path, output_base, label='eyes_closed')


Processing sub-NORB00001 ses-1 from sub-NORB00001_ses-1_task-EEG_eeg.edf

Loaded: sub-NORB00001_ses-1_task-EEG_eeg.edf
  Removed bad channels: ['Pg1', 'Pg2']

Extracted 32 valid segments
  Saved segment 1 matrix: sub-NORB00001_ses-1_segment-01_pearson_corr.csv (shape=(19, 19))
  Saved segment 2 matrix: sub-NORB00001_ses-1_segment-02_pearson_corr.csv (shape=(19, 19))
  Saved segment 3 matrix: sub-NORB00001_ses-1_segment-03_pearson_corr.csv (shape=(19, 19))
  Saved segment 4 matrix: sub-NORB00001_ses-1_segment-04_pearson_corr.csv (shape=(19, 19))
  Saved segment 5 matrix: sub-NORB00001_ses-1_segment-05_pearson_corr.csv (shape=(19, 19))
  Saved segment 6 matrix: sub-NORB00001_ses-1_segment-06_pearson_corr.csv (shape=(19, 19))
  Saved segment 7 matrix: sub-NORB00001_ses-1_segment-07_pearson_corr.csv (shape=(19, 19))
  Saved segment 8 matrix: sub-NORB00001_ses-1_segment-08_pearson_corr.csv (shape=(19, 19))
  Saved segment 9 matrix: sub-NORB00001_ses-1_segment-09_pearson_corr.csv (shape=(19

## Average across all segments
* Take absolute alues and then average it

In [8]:
# Create session-average absolute correlation matrices from segment files
segment_base = output_base
session_average_base = Path('/home/aloo/CNS_Summer_Project_1/correlation_matrices_session_abs_average')
session_average_base.mkdir(parents=True, exist_ok=True)


def save_session_average_matrix(matrix, channel_labels, subject_id, session_id, output_base):
    out_path = output_base / f'{subject_id}_{session_id}_avg_abs_pearson_corr.csv'
    pd.DataFrame(matrix, index=channel_labels, columns=channel_labels).to_csv(out_path)
    return out_path


def compute_session_average_matrices(segment_base, output_base):
    for subject_dir in sorted(segment_base.iterdir()):
        if not subject_dir.is_dir():
            continue
        for session_dir in sorted(subject_dir.iterdir()):
            if not session_dir.is_dir():
                continue

            matrices = []
            labels = None
            for csv_path in sorted(session_dir.glob('*.csv')):
                df = pd.read_csv(csv_path, index_col=0)
                matrices.append(np.abs(df.values))
                labels = df.index.tolist()

            if not matrices:
                print(f'No segment matrices found for {subject_dir.name} {session_dir.name}')
                continue

            avg_matrix = np.mean(matrices, axis=0)
            out_path = save_session_average_matrix(avg_matrix, labels, subject_dir.name, session_dir.name, output_base)
            print(f'Saved avg abs matrix for {subject_dir.name} {session_dir.name}: {out_path.name}')

    print(f'\nSession-average matrices saved under: {output_base}')


compute_session_average_matrices(segment_base, session_average_base)

Saved avg abs matrix for sub-NORB00001 ses-1: sub-NORB00001_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00003 ses-1: sub-NORB00003_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00005 ses-1: sub-NORB00005_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00006 ses-1: sub-NORB00006_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00007 ses-1: sub-NORB00007_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00008 ses-1: sub-NORB00008_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00009 ses-1: sub-NORB00009_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00010 ses-1: sub-NORB00010_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00013 ses-1: sub-NORB00013_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00014 ses-1: sub-NORB00014_ses-1_avg_abs_pearson_corr.csv
Saved avg abs matrix for sub-NORB00015 ses-1: sub-NORB00015_ses-1_avg_abs_pearson_corr.csv